In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
TARGET_PROMPT_ID = 3
MODEL = "gpt-5.4-nano-2026-03-17"

In [3]:
import polars as pl
from utils import load_asap_dataset, get_score_range

asap = load_asap_dataset("./dataset", essay_set=TARGET_PROMPT_ID)
asap = asap.sample(fraction=0.3, shuffle=True, seed=25).sort("essay_id")
asap

essay_set,essay_id,essay,score
i64,i64,str,i64
3,5985,"""The affects of the cyclist is …",0
3,5993,"""In the essay Do Not Exceed Pos…",2
3,5996,"""In the story “Rough Road Ahead…",1
3,5997,"""Some features of the setting a…",2
3,6008,"""The features of this setting a…",1
…,…,…,…
3,7700,"""The cyclist in this essay is t…",3
3,7701,"""It give him a hard time so he …",0
3,7704,"""In the story, the setting affe…",2


In [4]:
# Load llm prompts
with open("llm_prompts/vanilla_system.md", "r") as f:
    system_prompt = f.read()

with open("llm_prompts/vanilla_user.md", "r") as f:
    user_prompt = f.read()

# Load essay topic
with open(f"llm_prompts/asap_original/prompt_{TARGET_PROMPT_ID}.md", "r") as f:
    essay_topic = f.read()

In [13]:
minscore, maxscore = get_score_range("ASAP", TARGET_PROMPT_ID)

queries = {}
for id, essay in asap[["essay_id", "essay"]].iter_rows():
    queries[id] = [
        {
            "role": "system",
            "content": system_prompt.replace(
                "{{minimum score value}}", str(minscore)
            ).replace("{{maximum score value}}", str(maxscore)),
        },
        {
            "role": "user",
            "content": user_prompt.replace("{{prompt}}", essay_topic)
            .replace("{{essay}}", essay)
            .replace("{{minimum score value}}", str(minscore))
            .replace("{{maximum score value}}", str(maxscore)),
        },
    ]

In [14]:
import tiktoken


def count_total_tokens(queries, model="gpt-4o-mini"):
    enc = tiktoken.encoding_for_model(model)
    total_tokens = 0

    for qlist in queries.values():
        for msg in qlist:
            total_tokens += len(enc.encode(msg["content"]))

    return total_tokens


total = count_total_tokens(queries)
print(f"総トークン数: {total:,}")  # ← 3桁ごとにカンマ

総トークン数: 257,573


In [ ]:
from tokenrail import BatchExecutor, PerRequestJsonSink, ResultsJsonlSink, RailClient, RollingMetricsMonitor
from tokenrail.executor import batch_items_from_queries

client = RailClient.openai(max_retries=6)

items = batch_items_from_queries(
    queries,
    model=MODEL,
    reasoning_effort="medium",
    verbosity="low",
)

executor = BatchExecutor(
    client=client,
    max_workers=3,
    sinks=[PerRequestJsonSink(f"out/{MODEL}/{TARGET_PROMPT_ID}"), ResultsJsonlSink(f"out/{MODEL}/{TARGET_PROMPT_ID}/results.jsonl")],
    monitor=RollingMetricsMonitor(),
)

stats = executor.run(items)
print(stats.to_dict())

[1/517] id=5993 model=gpt-5.4-nano-2026-03-17 status=ok elapsed=00:00:03 eta=00:18:16 finish=21:27:00 in=528 cached=0 out=387 reasoning=51 total=915 rpm=1 tpm=915 cost=$0.000589 payer=developer developer_total=$0.000589 openai_total=$0.000000
[2/517] id=5996 model=gpt-5.4-nano-2026-03-17 status=ok elapsed=00:00:04 eta=00:11:12 finish=21:19:56 in=491 cached=0 out=410 reasoning=41 total=901 rpm=2 tpm=1816 cost=$0.000611 payer=developer developer_total=$0.001200 openai_total=$0.000000
[3/517] id=5985 model=gpt-5.4-nano-2026-03-17 status=ok elapsed=00:00:04 eta=00:07:33 finish=21:16:18 in=516 cached=0 out=489 reasoning=74 total=1005 rpm=3 tpm=2821 cost=$0.000714 payer=developer developer_total=$0.001914 openai_total=$0.000000
[4/517] id=6022 model=gpt-5.4-nano-2026-03-17 status=ok elapsed=00:00:06 eta=00:09:12 finish=21:17:58 in=538 cached=0 out=381 reasoning=0 total=919 rpm=4 tpm=3740 cost=$0.000584 payer=developer developer_total=$0.002498 openai_total=$0.000000
[5/517] id=6008 model=gpt

In [5]:
import re
import json

results = []
errors = []

with open(f"out/{MODEL}/{TARGET_PROMPT_ID}/results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        try:
            entry = json.loads(line.strip())
        except json.JSONDecodeError:
            continue  # JSONとして読み込めない行はスキップ

        entry_id = entry.get("id")
        text = entry.get("output_text", "")

        # <score>X</score> または Score: X （整数・小数対応、マイナスなし）
        match = re.search(r"<score>(\d+(?:\.\d+)?)</score>", text)
        if match:
            score = float(match.group(1))
            results.append({"id": entry_id, "score": score})
            continue

        match = re.search(r"Score:\s*(\d+(?:\.\d+)?)", text, re.IGNORECASE)
        if match:
            score = float(match.group(1))
            results.append({"id": entry_id, "score": score})
        else:
            errors.append(f"ID {entry_id} のスコアが不正な形式です: {text}")


print(f"抽出結果: {len(results)}件、エラー: {len(errors)}件")

抽出結果: 267件、エラー: 0件


In [6]:
errors

[]

In [7]:
results_df = pl.DataFrame(
    results, schema_overrides={"id": pl.Int64, "score": pl.Float64}
)
results_df = results_df.rename({"id": "essay_id", "score": "predicted_score"})
results_df = results_df.with_columns(pl.col("predicted_score").round(0).cast(pl.Int64))
results_df

essay_id,predicted_score
i64,i64
6242,1
6011,0
6227,2
6311,2
6221,2
…,…
6432,2
6465,1
6443,2


In [12]:
final_df = asap.join(results_df, on="essay_id", how="inner")
final_df = final_df.with_columns(pl.col("predicted_score").drop_nulls())
final_df

essay_set,essay_id,essay,score,predicted_score
i64,i64,str,i64,i64
3,5985,"""The affects of the cyclist is …",0,0
3,5993,"""In the essay Do Not Exceed Pos…",2,2
3,5996,"""In the story “Rough Road Ahead…",1,1
3,5997,"""Some features of the setting a…",2,1
3,6008,"""The features of this setting a…",1,2
…,…,…,…,…
3,7663,"""In “Rough Road Ahead: Do Not E…",3,2
3,7667,"""The features in the setting af…",2,1
3,7670,"""A response that explains how t…",0,0


In [13]:
from sklearn.metrics import cohen_kappa_score

minscore, maxscore = get_score_range("ASAP", TARGET_PROMPT_ID)
qwk = cohen_kappa_score(
    final_df["predicted_score"].to_numpy(),
    final_df["score"].to_numpy(),
    weights="quadratic",
    labels=list(range(minscore, maxscore + 1)),
)
print(f"QWK: {qwk:.4f}")

QWK: 0.3242
